# 00 - Environment Visual Probe

**Goal:** Build intuition for what SoccerTwos gives the agent before PPO or value functions enter the picture.

**What you will learn:** How to read the observation/action spaces, why rewards are sparse, and how random actions look in plots.

**Inputs:** An installed `soccertwos` environment and the organized project package.

**Outputs:** Tables for observations/actions plus reward, trajectory, and action-distribution plots.

**Success criteria:** A short headless rollout runs and the plots make the sparse-reward problem visible.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

PROJECT_MARKER = Path("soccer_twos_project") / "notebook_tools.py"


def _running_in_colab():
    if "google.colab" in sys.modules:
        return True
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"):
        return True
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _candidate_project_roots():
    seen = set()

    def add(path):
        path = Path(path).expanduser()
        key = str(path)
        if key not in seen:
            seen.add(key)
            yield path

    for env_name in ("SOCCER_TWOS_PROJECT_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(env_name)
        if value:
            yield from add(value)

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        yield from add(base)
        yield from add(base / "soccer-twos-starter")
        yield from add(base / "project" / "soccer-twos-starter")

    if sys.platform == "darwin":
        yield from add(
            Path.home()
            / "all_data"
            / "Georgia Tech"
            / "Course Content"
            / "CS 8803- DRL"
            / "project"
            / "soccer-twos-starter"
        )

    if _running_in_colab():
        try:
            from google.colab import drive  # type: ignore
            if not Path("/content/drive/MyDrive").exists():
                drive.mount("/content/drive")
        except Exception:
            pass
        for drive_root in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives"), Path("/content")):
            for relative in (
                Path("CS 8803- DRL") / "project" / "soccer-twos-starter",
                Path("project") / "soccer-twos-starter",
                Path("soccer-twos-starter"),
                Path("Colab Notebooks") / "soccer-twos-starter",
            ):
                yield from add(drive_root / relative)


def _find_project_root():
    for candidate in _candidate_project_roots():
        if (candidate / PROJECT_MARKER).exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find soccer_twos_project/notebook_tools.py. "
        "Open this notebook from the project root/notebooks folder, or set SOCCER_TWOS_PROJECT_ROOT."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for _module_name in list(sys.modules):
    if _module_name == "soccer_twos_project" or _module_name.startswith("soccer_twos_project."):
        del sys.modules[_module_name]

importlib.invalidate_caches()
from IPython.display import Markdown, display
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

LEARNING_DIR = learning_artifact_dir(ctx, "visual_learning")
print("Learning artifacts:", LEARNING_DIR)

## Environment Sanity Setup

The training notebooks use a simplified single-player environment: one observation vector and one flat action index. The final live match still has four players and branched actions.

In [ ]:
try:
    spaces = inspect_environment_spaces()
except Exception as exc:
    spaces = None
    print("Environment inspection failed. Activate the soccertwos environment, then rerun this cell.")
    print(type(exc).__name__ + ":", exc)

## What The Agent Sees

The observation is a 336-number vector. We do not hand-code what each number means during PPO; the neural network learns patterns from the vector. For a beginner check, inspect its shape, scale, and a few entries.

In [ ]:
import numpy as np
import pandas as pd

observation_table = None
try:
    from soccer_twos import EnvType
    env = make_soccer_env(
        render=False,
        variation=EnvType.team_vs_policy,
        flatten_branched=True,
        single_player=True,
    )
    try:
        obs = np.asarray(env.reset(), dtype=np.float32).reshape(-1)
    finally:
        env.close()
    observation_table = pd.DataFrame({"index": np.arange(len(obs)), "value": obs})
    display(observation_table["value"].describe().to_frame("observation_value"))
    display(observation_table.head(20))
except Exception as exc:
    print("Observation snapshot skipped:", type(exc).__name__, exc)

## What The Agent Can Do

Unity receives three branches: move forward/back, strafe, and rotate. RLlib sees a flattened `Discrete(27)` action during single-player PPO training.

In [ ]:
print_soccer_action_help()
display(soccer_action_glossary())
display(soccer_flat_action_table().head(12))
display(soccer_named_action_table())

## Random Rollout

A random policy samples one of the 27 actions at each step. Rewards are usually zero because goals are rare, so the plots also track distances to show whether anything task-relevant is happening.

In [ ]:
RENDER_UNITY = False
ROLLOUT_STEPS = 200

resolved_render_unity = resolve_unity_render_request(
    RENDER_UNITY,
    ctx=ctx,
    label="Random rollout Unity playback",
)

random_rollout = None
try:
    random_rollout = collect_single_player_rollout(
        policy="random",
        steps=ROLLOUT_STEPS,
        render=resolved_render_unity,
        label="random policy",
    )
    display(rollout_summary_table({"random": random_rollout}))
    action_cols = [
        "step", "flat_action", "branch_action", "action_meaning",
        "reward", "player_ball_dist", "ball_goal_dist",
    ]
    display(random_rollout[[col for col in action_cols if col in random_rollout.columns]].head(15))
except Exception as exc:
    print("Random rollout skipped:", type(exc).__name__, exc)


In [ ]:
if random_rollout is not None and not random_rollout.empty:
    plot_reward_timeline(random_rollout, title="Random policy: reward timeline");
    plot_rollout_overview(random_rollout, title="Random policy: reward, distances, actions");
    plot_top_down_trajectory(random_rollout, title="Random policy: top-down movement");
    plot_action_distribution(random_rollout, title="Random policy: action distribution");

## Manual Fixed-Action Rollout

A fixed action is not a good policy. It is a tiny experiment for learning what one action does physically.

In [ ]:
FIXED_POLICY = "forward"  # Try: noop, forward, backward, right, left, rotate_a, rotate_d, or an integer 0-26.
fixed_rollout = None
try:
    print("Fixed policy decoded:")
    print_json(soccer_action_description(FIXED_POLICY))
    fixed_rollout = collect_single_player_rollout(
        policy=FIXED_POLICY,
        steps=ROLLOUT_STEPS,
        render=False,
        label="fixed {}".format(FIXED_POLICY),
    )
    display(rollout_summary_table({"random": random_rollout, "fixed {}".format(FIXED_POLICY): fixed_rollout}))
    plot_rollout_comparison(
        {"random": random_rollout, "fixed {}".format(FIXED_POLICY): fixed_rollout},
        title="Random vs fixed-action behavior",
    );
    plot_top_down_trajectory(fixed_rollout, title="Fixed action trajectory");
except Exception as exc:
    print("Fixed-action rollout skipped:", type(exc).__name__, exc)

## Why Reward Is Sparse

SoccerTwos gives meaningful task reward mostly around scoring and conceding. Early policies can move for many steps without scoring, so reward curves alone may look flat. Distance and trajectory plots help you see whether behavior is changing before wins appear.

## Key Takeaways

The agent sees a fixed-size vector, chooses from 27 flattened actions during training, and usually receives zero reward at first. Visual rollout metrics make early behavior easier to debug than reward alone.

## What To Run Next

Run `01_rl_basics_policy_value_advantage.ipynb` to connect these rollouts to return, value, advantage, and PPO.